CSE 144 Final Project — Vit + EfficientNet ensemble

- Two backbones: ViT-B/16 (SWAG weights) plus efficientnet-b3 each trained independently
- CutMix + Mixup randomly alternated each batch
- 8 TTA views per image (up from 4)
- Ensemble: average softmax probabilities from both models
- Label smoothing reduced to 0.0 which helps on tiny datasets
- All fixes from v2 retained numeric label mapping, full 1000-image Stage 3, cosine LR and differential LR

1. Setup & Seed

In [ ]:
import os
import random
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset, Dataset
from torchvision import datasets, transforms
from torchvision.models import (
    efficientnet_b3, EfficientNet_B3_Weights,
    vit_b_16,        ViT_B_16_Weights,
)
from sklearn.model_selection import train_test_split


def set_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark     = False
    torch.backends.cudnn.deterministic = True

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

os.makedirs("./checkpoints", exist_ok=True)

2. Download dataset

In [ ]:
import kagglehub

path = kagglehub.competition_download(
    "ucsc-cse-144-spring-2026-final-project"
)

print("Competition path:", path)

train_data_path = os.path.join(path, "train")
test_data_path  = os.path.join(path, "test")

3. Numeric Label Mapping

In [ ]:
class NumericImageFolder(datasets.ImageFolder):
    def find_classes(self, directory):
        classes = sorted(
            [e.name for e in os.scandir(directory) if e.is_dir()],
            key=lambda x: int(x)
        )
        class_to_idx = {cls: int(cls) for cls in classes}
        return classes, class_to_idx


base_dataset = NumericImageFolder(train_data_path, transform=None)
num_classes  = len(base_dataset.classes)

print(f"Training images: {len(base_dataset)} | Classes: {num_classes}")
print("First 5:", list(base_dataset.class_to_idx.items())[:5])
print("Last  5:", list(base_dataset.class_to_idx.items())[-5:])

assert num_classes == 100
assert base_dataset.class_to_idx["0"]  == 0
assert base_dataset.class_to_idx["10"] == 10
assert base_dataset.class_to_idx["99"] == 99
print("Label mapping OK.")

4. Transforms and tta

vit-b/16 expects 224×224. We use the same size for EfficientNet-B3 to keep loaders identical.

tta: 8 views per image — center crop, h-flip, two random crops at different scales, two larger resizes, v-flip, rotation.

In [ ]:
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

def norm():
    return transforms.Normalize(mean=MEAN, std=STD)

train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(384, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.1),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    norm(),
])

val_transforms = transforms.Compose([
    transforms.Resize(384),
    transforms.CenterCrop(384),
    transforms.ToTensor(),
    norm(),
])

tta_transforms = [
    transforms.Compose([transforms.Resize(384), transforms.CenterCrop(384), transforms.ToTensor(), norm()]),
    transforms.Compose([transforms.Resize(384), transforms.CenterCrop(384), transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor(), norm()]),
    transforms.Compose([transforms.RandomResizedCrop(384, scale=(0.85, 1.0)), transforms.ToTensor(), norm()]),
    transforms.Compose([transforms.RandomResizedCrop(384, scale=(0.85, 1.0)), transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor(), norm()]),
    transforms.Compose([transforms.Resize(384), transforms.CenterCrop(384), transforms.ToTensor(), norm()]),
    transforms.Compose([transforms.Resize(384), transforms.CenterCrop(384), transforms.ToTensor(), norm()]),
    transforms.Compose([transforms.Resize(384), transforms.CenterCrop(384), transforms.RandomVerticalFlip(p=1.0), transforms.ToTensor(), norm()]),
    transforms.Compose([transforms.Resize(384), transforms.CenterCrop(384), transforms.RandomRotation(degrees=(10, 10)), transforms.ToTensor(), norm()]),
]

print(f"TTA views: {len(tta_transforms)}")

5. Data Loaders

In [ ]:
targets = [label for _, label in base_dataset.samples]
indices = np.arange(len(base_dataset))

train_idx, val_idx = train_test_split(
    indices, test_size=0.20, random_state=42, stratify=targets
)

_train_ds = NumericImageFolder(train_data_path, transform=train_transforms)
_val_ds   = NumericImageFolder(train_data_path, transform=val_transforms)

train_subset = Subset(_train_ds, train_idx)
val_subset   = Subset(_val_ds,   val_idx)

full_train_dataset = NumericImageFolder(train_data_path, transform=train_transforms)

BATCH_SIZE = 32

tune_train_loader = DataLoader(train_subset,      batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
tune_val_loader   = DataLoader(val_subset,         batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
full_train_loader = DataLoader(full_train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)

print(f"Tune train: {len(train_subset)} | Tune val: {len(val_subset)} | Full: {len(full_train_dataset)}")

6. Model Builders

Two separate factory functions — one for each backbone.

In [ ]:
def build_efficientnet(num_classes=100):
    """EfficientNet-B3 with dropout head."""
    model = efficientnet_b3(weights=EfficientNet_B3_Weights.IMAGENET1K_V1)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features, num_classes)
    )
    return model


def build_vit(num_classes=100):
    """
    ViT-B/16 with SWAG E2E weights — these are stronger than standard
    IMAGENET1K_V1 weights because SWAG was trained on a much larger weakly-
    supervised dataset and then fine-tuned on ImageNet.
    Falls back to standard weights if SWAG is unavailable.
    """
    try:
        model = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_SWAG_E2E_V1)
        print("ViT loaded with SWAG_E2E weights.")
    except Exception:
        model = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
        print("ViT loaded with standard IMAGENET1K_V1 weights.")

    in_features = model.heads.head.in_features
    model.heads.head = nn.Linear(in_features, num_classes)
    return model


_m = build_efficientnet(100)
print(f"EfficientNet-B3 params: {sum(p.numel() for p in _m.parameters())/1e6:.1f}M")
del _m

_m = build_vit(100)
print(f"ViT-B/16 params:        {sum(p.numel() for p in _m.parameters())/1e6:.1f}M")
del _m

7. Mixup + CutMix

In [ ]:
def mixup_data(x, y, alpha=0.2):
    lam   = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    index = torch.randperm(x.size(0), device=x.device)
    mixed = lam * x + (1 - lam) * x[index]
    return mixed, y, y[index], lam


def cutmix_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    B, C, H, W = x.shape
    index = torch.randperm(B, device=x.device)

    cut_ratio = np.sqrt(1 - lam)
    cut_h = int(H * cut_ratio)
    cut_w = int(W * cut_ratio)

    cx = np.random.randint(W)
    cy = np.random.randint(H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    y2 = np.clip(cy + cut_h // 2, 0, H)

    mixed = x.clone()
    mixed[:, :, y1:y2, x1:x2] = x[index, :, y1:y2, x1:x2]

    lam = 1 - (x2 - x1) * (y2 - y1) / (W * H)
    return mixed, y, y[index], lam


def mixed_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

8. Training Helpers

In [ ]:
def run_one_epoch(model, loader, criterion, optimizer=None, scheduler=None,
                  use_augmix=False, mixup_alpha=0.2, cutmix_alpha=1.0):
    is_training = optimizer is not None
    model.train() if is_training else model.eval()

    total_loss, total_correct, total_images = 0.0, 0, 0

    ctx = torch.enable_grad() if is_training else torch.no_grad()
    with ctx:
        for images, labels in tqdm(loader, leave=False):
            images = images.to(device)
            labels = labels.to(device)

            if is_training and use_augmix:
                if random.random() < 0.5:
                    images, y_a, y_b, lam = mixup_data(images, labels, alpha=mixup_alpha)
                else:
                    images, y_a, y_b, lam = cutmix_data(images, labels, alpha=cutmix_alpha)
                outputs = model(images)
                loss    = mixed_criterion(criterion, outputs, y_a, y_b, lam)
            else:
                outputs = model(images)
                loss    = criterion(outputs, labels)

            if is_training:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()

            total_loss    += loss.item() * images.size(0)
            total_correct += (outputs.argmax(dim=1) == labels).sum().item()
            total_images  += labels.size(0)

    if is_training and scheduler is not None:
        scheduler.step()

    return total_loss / total_images, total_correct / total_images


def fit(model, train_loader, val_loader, optimizer, scheduler, epochs,
        stage_name, criterion, best_val_acc=0.0, use_augmix=False,
        mixup_alpha=0.2, cutmix_alpha=1.0, patience=7, ckpt_path="./checkpoints/best_model.pt"):

    patience_counter = 0
    history = []

    for epoch in range(1, epochs + 1):
        print(f"\n{stage_name} | Epoch {epoch}/{epochs}")

        train_loss, train_acc = run_one_epoch(
            model, train_loader, criterion, optimizer, scheduler,
            use_augmix=use_augmix, mixup_alpha=mixup_alpha, cutmix_alpha=cutmix_alpha
        )
        val_loss, val_acc = run_one_epoch(model, val_loader, criterion)

        history.append(dict(
            stage=stage_name, epoch=epoch,
            train_loss=train_loss, train_acc=train_acc,
            val_loss=val_loss,   val_acc=val_acc
        ))

        lr_now = optimizer.param_groups[0]["lr"]
        print(f"  Train {train_acc*100:.2f}% loss={train_loss:.4f} | "
              f"Val {val_acc*100:.2f}% loss={val_loss:.4f} | LR={lr_now:.2e}")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), ckpt_path)
            print(f"  ✓ Best: {best_val_acc*100:.2f}% saved → {ckpt_path}")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("  Early stopping.")
                break

    return best_val_acc, history


Model A: EfficientNet-B3

9A. Build + Stage 1 (head only)

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=0.0)

effnet = build_efficientnet(num_classes).to(device)

for param in effnet.features.parameters():
    param.requires_grad = False
for param in effnet.classifier.parameters():
    param.requires_grad = True

S1 = 10
opt_en_s1  = torch.optim.AdamW(filter(lambda p: p.requires_grad, effnet.parameters()), lr=1e-3, weight_decay=1e-4)
sched_en_s1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt_en_s1, T_max=S1, eta_min=1e-5)

best_val_effnet = 0.0
all_history_effnet = []

best_val_effnet, h = fit(
    effnet, tune_train_loader, tune_val_loader,
    opt_en_s1, sched_en_s1, S1,
    stage_name="[EffNet] Stage 1: head only",
    criterion=criterion,
    best_val_acc=best_val_effnet,
    use_augmix=False,
    ckpt_path="./checkpoints/best_effnet.pt"
)
all_history_effnet.extend(h)
print(f"\nEffNet best val after S1: {best_val_effnet*100:.2f}%")

9B. EfficientNet Stage 2 — Unfreeze Last 3 Blocks + Mixup/CutMix

In [ ]:
effnet.load_state_dict(torch.load("./checkpoints/best_effnet.pt", map_location=device))

for param in effnet.parameters():
    param.requires_grad = False
for block in effnet.features[-3:]:
    for param in block.parameters():
        param.requires_grad = True
for param in effnet.classifier.parameters():
    param.requires_grad = True

backbone_p   = [p for p in effnet.features[-3:].parameters() if p.requires_grad]
classifier_p = list(effnet.classifier.parameters())

S2 = 20
opt_en_s2 = torch.optim.AdamW([
    {"params": backbone_p,   "lr": 3e-5, "weight_decay": 1e-4},
    {"params": classifier_p, "lr": 3e-4, "weight_decay": 1e-4},
])
sched_en_s2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt_en_s2, T_max=S2, eta_min=1e-7)

best_val_effnet, h = fit(
    effnet, tune_train_loader, tune_val_loader,
    opt_en_s2, sched_en_s2, S2,
    stage_name="[EffNet] Stage 2: last 3 blocks + Mixup/CutMix",
    criterion=criterion,
    best_val_acc=best_val_effnet,
    use_augmix=True, mixup_alpha=0.2, cutmix_alpha=1.0,
    patience=7,
    ckpt_path="./checkpoints/best_effnet.pt"
)
all_history_effnet.extend(h)
print(f"\nEffNet best val after S2: {best_val_effnet*100:.2f}%")

9C. EfficientNet Stage 3 — Full 1000-Image Fine-Tune

In [ ]:
effnet.load_state_dict(torch.load("./checkpoints/best_effnet.pt", map_location=device))

for param in effnet.parameters():
    param.requires_grad = False
for block in effnet.features[-3:]:
    for param in block.parameters():
        param.requires_grad = True
for param in effnet.classifier.parameters():
    param.requires_grad = True

backbone_p   = [p for p in effnet.features[-3:].parameters() if p.requires_grad]
classifier_p = list(effnet.classifier.parameters())

S3 = 15
opt_en_s3 = torch.optim.AdamW([
    {"params": backbone_p,   "lr": 1e-5, "weight_decay": 1e-4},
    {"params": classifier_p, "lr": 1e-4, "weight_decay": 1e-4},
])
sched_en_s3 = torch.optim.lr_scheduler.CosineAnnealingLR(opt_en_s3, T_max=S3, eta_min=1e-7)

print("[EffNet] Stage 3: fine-tuning on all 1000 images...")
for epoch in range(1, S3 + 1):
    train_loss, train_acc = run_one_epoch(
        effnet, full_train_loader, criterion, opt_en_s3, sched_en_s3,
        use_augmix=True, mixup_alpha=0.15, cutmix_alpha=1.0
    )
    print(f"  Epoch {epoch}/{S3} | Train {train_acc*100:.2f}% loss={train_loss:.4f} "
          f"LR={opt_en_s3.param_groups[0]['lr']:.2e}")

torch.save(effnet.state_dict(), "./checkpoints/final_effnet.pt")
print("Saved ./checkpoints/final_effnet.pt")


Model B: ViT-B/16

10A. Build + Stage 1 (head only)

For vit we freeze `model.encoder` and train only `model.heads`.

In [ ]:
set_seed(42)

vit = build_vit(num_classes).to(device)

for param in vit.encoder.parameters():
    param.requires_grad = False
for param in vit.conv_proj.parameters():
    param.requires_grad = False
for param in vit.heads.parameters():
    param.requires_grad = True

S1 = 10
opt_vit_s1  = torch.optim.AdamW(filter(lambda p: p.requires_grad, vit.parameters()), lr=1e-3, weight_decay=1e-4)
sched_vit_s1 = torch.optim.lr_scheduler.CosineAnnealingLR(opt_vit_s1, T_max=S1, eta_min=1e-5)

best_val_vit = 0.0
all_history_vit = []

best_val_vit, h = fit(
    vit, tune_train_loader, tune_val_loader,
    opt_vit_s1, sched_vit_s1, S1,
    stage_name="[ViT] Stage 1: head only",
    criterion=criterion,
    best_val_acc=best_val_vit,
    use_augmix=False,
    ckpt_path="./checkpoints/best_vit.pt"
)
all_history_vit.extend(h)
print(f"\nViT best val after S1: {best_val_vit*100:.2f}%")

10B. ViT Stage 2 — Unfreeze Last 3 Transformer Blocks plus Mixup CutMix

In [ ]:
vit.load_state_dict(torch.load("./checkpoints/best_vit.pt", map_location=device))

for param in vit.parameters():
    param.requires_grad = False

for layer in vit.encoder.layers[-3:]:
    for param in layer.parameters():
        param.requires_grad = True

for param in vit.encoder.ln.parameters():
    param.requires_grad = True

for param in vit.heads.parameters():
    param.requires_grad = True

encoder_p = [p for name, p in vit.named_parameters()
             if p.requires_grad and "heads" not in name]
head_p    = list(vit.heads.parameters())

S2 = 20
opt_vit_s2 = torch.optim.AdamW([
    {"params": encoder_p, "lr": 1e-5, "weight_decay": 1e-4},
    {"params": head_p,    "lr": 1e-4, "weight_decay": 1e-4},
])
sched_vit_s2 = torch.optim.lr_scheduler.CosineAnnealingLR(opt_vit_s2, T_max=S2, eta_min=1e-7)

best_val_vit, h = fit(
    vit, tune_train_loader, tune_val_loader,
    opt_vit_s2, sched_vit_s2, S2,
    stage_name="[ViT] Stage 2: last 3 blocks + Mixup/CutMix",
    criterion=criterion,
    best_val_acc=best_val_vit,
    use_augmix=True, mixup_alpha=0.2, cutmix_alpha=1.0,
    patience=7,
    ckpt_path="./checkpoints/best_vit.pt"
)
all_history_vit.extend(h)
print(f"\nViT best val after S2: {best_val_vit*100:.2f}%")

10C. ViT Stage 3 — Full 1000-Image Fine-Tune

In [ ]:
vit.load_state_dict(torch.load("./checkpoints/best_vit.pt", map_location=device))

for param in vit.parameters():
    param.requires_grad = False
for layer in vit.encoder.layers[-3:]:
    for param in layer.parameters():
        param.requires_grad = True
for param in vit.encoder.ln.parameters():
    param.requires_grad = True
for param in vit.heads.parameters():
    param.requires_grad = True

encoder_p = [p for name, p in vit.named_parameters()
             if p.requires_grad and "heads" not in name]
head_p    = list(vit.heads.parameters())

S3 = 15
opt_vit_s3 = torch.optim.AdamW([
    {"params": encoder_p, "lr": 5e-6, "weight_decay": 1e-4},
    {"params": head_p,    "lr": 5e-5, "weight_decay": 1e-4},
])
sched_vit_s3 = torch.optim.lr_scheduler.CosineAnnealingLR(opt_vit_s3, T_max=S3, eta_min=1e-7)

print("[ViT] Stage 3: fine-tuning on all 1000 images...")
for epoch in range(1, S3 + 1):
    train_loss, train_acc = run_one_epoch(
        vit, full_train_loader, criterion, opt_vit_s3, sched_vit_s3,
        use_augmix=True, mixup_alpha=0.15, cutmix_alpha=1.0
    )
    print(f"  Epoch {epoch}/{S3} | Train {train_acc*100:.2f}% loss={train_loss:.4f} "
          f"LR={opt_vit_s3.param_groups[0]['lr']:.2e}")

torch.save(vit.state_dict(), "./checkpoints/final_vit.pt")
print("Saved ./checkpoints/final_vit.pt")

11. Save Checkpoints & History

In [ ]:
idx_to_class = {v: k for k, v in full_train_dataset.class_to_idx.items()}

torch.save({
    "model_state_dict": effnet.state_dict(),
    "num_classes":      num_classes,
    "class_to_idx":     full_train_dataset.class_to_idx,
    "idx_to_class":     idx_to_class,
    "best_val_acc":     best_val_effnet,
    "arch":             "efficientnet_b3",
}, "./checkpoints/checkpoint_effnet.pt")

torch.save({
    "model_state_dict": vit.state_dict(),
    "num_classes":      num_classes,
    "class_to_idx":     full_train_dataset.class_to_idx,
    "idx_to_class":     idx_to_class,
    "best_val_acc":     best_val_vit,
    "arch":             "vit_b_16",
}, "./checkpoints/checkpoint_vit.pt")

all_history = all_history_effnet + all_history_vit
pd.DataFrame(all_history).to_csv("training_history.csv", index=False)

print(f"EfficientNet-B3 best val: {best_val_effnet*100:.2f}%")
print(f"ViT-B/16        best val: {best_val_vit*100:.2f}%")
print("Checkpoints and history saved.")

12. Test Dataset

In [ ]:
class TestImageDataset(Dataset):
    def __init__(self, root, transform=None):
        self.root      = root
        self.transform = transform
        self.image_paths = sorted(
            [f for f in os.listdir(root) if f.lower().endswith((".png", ".jpg", ".jpeg"))],
            key=lambda x: int(os.path.splitext(x)[0])
        )

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path  = os.path.join(self.root, self.image_paths[idx])
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, self.image_paths[idx]


probe = TestImageDataset(test_data_path)
print(f"Test images: {len(probe)}")

13. TTA Inference Helper

In [ ]:
def get_tta_probs(model, test_data_path, tta_transforms, num_classes, batch_size=32):
    """
    Runs TTA over all views and returns accumulated softmax probabilities
    shaped (N, num_classes) on CPU.
    """
    model.eval()
    n_test = len(TestImageDataset(test_data_path))
    accumulated = torch.zeros(n_test, num_classes)

    for tta_idx, tfm in enumerate(tta_transforms):
        print(f"  TTA view {tta_idx+1}/{len(tta_transforms)}...", end=" ")
        ds     = TestImageDataset(test_data_path, transform=tfm)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)

        offset = 0
        with torch.no_grad():
            for images, _ in tqdm(loader, leave=False):
                print("batch shape:", images.shape)
                images = images.to(device)
                probs  = F.softmax(model(images), dim=1).cpu()
                bsz    = images.size(0)
                accumulated[offset:offset+bsz] += probs
                offset += bsz
        print(f"done (offset={offset})")

    return accumulated

14. Run TTA on Both Models

In [ ]:
ckpt_en = torch.load("./checkpoints/checkpoint_effnet.pt", map_location=device)
effnet  = build_efficientnet(ckpt_en["num_classes"]).to(device)
effnet.load_state_dict(torch.load("./checkpoints/final_effnet.pt", map_location=device))

ckpt_vit = torch.load("./checkpoints/checkpoint_vit.pt", map_location=device)
vit      = build_vit(ckpt_vit["num_classes"]).to(device)
vit.load_state_dict(torch.load("./checkpoints/final_vit.pt", map_location=device))

idx_to_class = ckpt_en["idx_to_class"]

print("=== EfficientNet-B3 TTA ===")
probs_effnet = get_tta_probs(effnet, test_data_path, tta_transforms, num_classes)

print("\n=== ViT-B/16 TTA ===")
probs_vit = get_tta_probs(vit, test_data_path, tta_transforms, num_classes)

print(f"\nEffNet probs shape: {probs_effnet.shape}")
print(f"ViT    probs shape: {probs_vit.shape}")


15. Final Predictions

We try both equal-weight and ViT-weighted ensembles and pick the one with higher validation accuracy. Adjust `vit_weight` between 0.4–0.7 based on which model scored higher on your validation set.

In [ ]:
if best_val_vit >= best_val_effnet:
    vit_weight    = 0.6
    effnet_weight = 0.4
else:
    vit_weight    = 0.4
    effnet_weight = 0.6

print(f"Ensemble weights — ViT: {vit_weight}  EffNet: {effnet_weight}")

ensemble_probs = effnet_weight * probs_effnet + vit_weight * probs_vit
final_preds    = ensemble_probs.argmax(dim=1).numpy()

all_filenames = TestImageDataset(test_data_path).image_paths
all_labels    = [int(idx_to_class[int(pred)]) for pred in final_preds]

print(f"Predictions: {len(all_labels)}")
print("First 10 files:",  all_filenames[:10])
print("First 10 labels:", all_labels[:10])
print(f"Label range: {min(all_labels)} – {max(all_labels)} | Unique: {len(set(all_labels))}")

16. Create Submission

In [ ]:
submission_df = pd.DataFrame({"ID": all_filenames, "Label": all_labels})
submission_df.to_csv("submission.csv", index=False)

print(submission_df.head(10))
print(f"\nSaved submission.csv ({len(submission_df)} rows)")

17. Optional: Validate Ensemble on Held-Out Val Set

Run this to get a sense of how much the ensemble helps before submitting.

In [ ]:
def eval_ensemble_on_val(effnet, vit, val_subset, vit_weight=0.5):
    effnet.eval()
    vit.eval()

    loader = DataLoader(val_subset, batch_size=32, shuffle=False, num_workers=0)
    correct_ensemble  = 0
    correct_effnet    = 0
    correct_vit       = 0
    total = 0

    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Val ensemble"):
            images = images.to(device)
            labels = labels.to(device)

            p_en  = F.softmax(effnet(images), dim=1)
            p_vit = F.softmax(vit(images),   dim=1)
            p_ens = (1 - vit_weight) * p_en + vit_weight * p_vit

            correct_effnet   += (p_en.argmax(1)  == labels).sum().item()
            correct_vit      += (p_vit.argmax(1) == labels).sum().item()
            correct_ensemble += (p_ens.argmax(1) == labels).sum().item()
            total += labels.size(0)

    print(f"Val accuracy — EffNet: {correct_effnet/total*100:.2f}% | "
          f"ViT: {correct_vit/total*100:.2f}% | "
          f"Ensemble ({vit_weight:.1f}/{1-vit_weight:.1f}): {correct_ensemble/total*100:.2f}%")

_val_ds_check = NumericImageFolder(train_data_path, transform=val_transforms)
_val_subset_check = Subset(_val_ds_check, val_idx)

eval_ensemble_on_val(effnet, vit, _val_subset_check, vit_weight=vit_weight)